# How to catch a Kubernetes node pressure event before it takes down your pods

Requires Elasticsearch and Kibana 9.5+ with machine learning enabled, and the lab in `k8s/` (see `k8s/README.md`). Copy `.env.example` to `.env` and set an API key created in Kibana **Dev Tools**:

```
POST /_security/api_key
{ "name": "node-pressure-notebook", "role_descriptors": {} }
```

Order: `k8s/lab.sh up` and `k8s/lab.sh probe`, run sections 1 to 5, wait two hours, `k8s/lab.sh stage1`, wait 4.5 minutes, `k8s/lab.sh stage2`, wait for the evictions, then run sections 6 to 9. Section 10 cleans up.

In [ ]:
%pip install -q -r requirements.txt

## 1. Connect

In [ ]:
import json
import os
import time
from datetime import datetime, timedelta, timezone

import requests
from dotenv import load_dotenv
from elasticsearch import Elasticsearch

load_dotenv()

ES_URL = os.environ["ELASTICSEARCH_URL"]
API_KEY = os.environ["ELASTICSEARCH_API_KEY"]
KIBANA_URL = os.environ["KIBANA_URL"].rstrip("/")
SPACE = os.getenv("KIBANA_SPACE", "default")
KBN_BASE = KIBANA_URL if SPACE == "default" else f"{KIBANA_URL}/s/{SPACE}"

CLUSTER = os.getenv("LAB_CLUSTER_NAME", "node-pressure-v2")
NODE = os.getenv("LAB_NODE_NAME", "node-pressure-v2-worker")
TEST_ID = os.getenv("LAB_TEST_ID", "node-pressure-v2")
EVICTION_GIB = float(os.getenv("LAB_EVICTION_GIB", "6") or "6")
HEADROOM_GIB = float(os.getenv("LAB_HEADROOM_GIB", "6.5") or "6.5")
WINDOW_MINUTES = int(os.getenv("LAB_WINDOW_MINUTES", "60") or "60")
RANGE_FROM = os.getenv("LAB_RANGE_FROM", "").strip()
RANGE_TO = os.getenv("LAB_RANGE_TO", "").strip()
BASELINE_FROM = os.getenv("LAB_BASELINE_FROM", "").strip()

# Data streams written by the EDOT Collector and the k6 probe.
KUBELET_STREAM = "metrics-kubeletstatsreceiver.otel-*"
CLUSTER_STREAM = "metrics-k8sclusterreceiver.otel-*"
EVENTS_STREAM = "logs-k8seventsreceiver.otel-*"
PROBE_STREAM = "metrics-generic.otel-*"

# Created by this notebook and deleted in Clean up.
JOB_ID = os.getenv("LAB_JOB_ID", "node-pressure-memory-v2")
DATAFEED_ID = f"datafeed-{JOB_ID}"
LAB_TAG = "node-pressure-lab"
RULE_ANOMALY = "Node pressure v2: unusual memory headroom"
RULE_HEADROOM = "Node pressure v2: memory headroom warning"
RULE_MEMORY_PRESSURE = "[Kubernetes OTel] Node memory pressure"
RULE_DISK_PRESSURE = "[Kubernetes OTel] Node disk pressure"
SEVERITY = 25

es = Elasticsearch(ES_URL, api_key=API_KEY, request_timeout=120)
KBN_HEADERS = {
    "Authorization": f"ApiKey {API_KEY}",
    "kbn-xsrf": "true",
    "Content-Type": "application/json",
}


def kbn(method, path, **kwargs):
    """Call a Kibana API in the configured space and raise on HTTP errors."""
    response = requests.request(
        method, f"{KBN_BASE}{path}", headers=KBN_HEADERS, timeout=120, **kwargs
    )
    response.raise_for_status()
    return response.json() if response.text else {}


def esql(query, title=None):
    """Run an ES|QL query and print the result as a text table."""
    response = es.esql.query(query=query, format="txt")
    if title:
        print(f"\n{title}")
    print(response.body)
    return response.body


def esql_rows(query):
    """Run an ES|QL query and return a list of dicts."""
    response = es.esql.query(query=query)
    columns = [c["name"] for c in response["columns"]]
    return [dict(zip(columns, row)) for row in response["values"]]


def window(minutes=None):
    """WHERE fragment for the scenario window: fixed bounds from .env, or the last N minutes."""
    if RANGE_FROM and RANGE_TO:
        return f'@timestamp >= "{RANGE_FROM}" AND @timestamp <= "{RANGE_TO}"'
    return f"@timestamp >= NOW() - {minutes or WINDOW_MINUTES} minutes"


def parse_ts(value):
    """ISO timestamp or epoch milliseconds to an aware datetime."""
    if value is None:
        return None
    if isinstance(value, (int, float)):
        return datetime.fromtimestamp(value / 1000, tz=timezone.utc)
    return datetime.fromisoformat(str(value).replace("Z", "+00:00"))


def fmt(ts):
    return ts.strftime("%H:%M:%S") if ts else "none"


print(
    f"elasticsearch {es.info()['version']['number']} | space={SPACE} | window: {window()}"
)
kibana_check = requests.get(
    f"{KBN_BASE}/api/alerting/rules/_find",
    headers=KBN_HEADERS,
    params={"per_page": 1},
    timeout=60,
)
if kibana_check.status_code in (401, 403):
    raise RuntimeError(
        "The API key has no Kibana privileges, which the rules, the job sync and the dashboard need. "
        'Create one in Kibana Dev Tools with "role_descriptors": {} and put it in .env.'
    )
kibana_check.raise_for_status()
streams = es.indices.get_data_stream(name="*.otel-*")["data_streams"]
names = sorted(s["name"] for s in streams)
missing = []
for pattern in (KUBELET_STREAM, CLUSTER_STREAM, EVENTS_STREAM, PROBE_STREAM):
    prefix = pattern.rstrip("*")
    found = [n for n in names if n.startswith(prefix)]
    print(f"  {pattern}: {'ok ' + ', '.join(found) if found else 'MISSING'}")
    if not found:
        missing.append(pattern)
if missing:
    raise RuntimeError(
        f"missing {missing}: run ./lab.sh up and ./lab.sh probe, wait two minutes, and rerun this cell"
    )

## 2. Check the node metrics and conditions

Available memory per node, the MemoryPressure and DiskPressure conditions (0 = false, 1 = true), and the first timestamp in the node history.

In [ ]:
esql(
    f"""
FROM {KUBELET_STREAM}
| WHERE k8s.cluster.name == "{CLUSTER}"
    AND k8s.node.memory.available IS NOT NULL
    AND @timestamp > NOW() - 5 minutes
| STATS available_gib = ROUND(MAX(k8s.node.memory.available) / 1073741824.0, 2),
    latest_sample = MAX(@timestamp), samples = COUNT(*)
    BY k8s.node.name
| LIMIT 20
""",
    "Available memory per node (last 5 minutes)",
)

esql(
    f"""
FROM {CLUSTER_STREAM}
| WHERE k8s.cluster.name == "{CLUSTER}"
    AND @timestamp > NOW() - 5 minutes
| STATS memory_pressure = MAX(k8s.node.condition_memory_pressure),
    disk_pressure = MAX(k8s.node.condition_disk_pressure),
    latest_sample = MAX(@timestamp)
    BY k8s.cluster.name, k8s.node.name
| LIMIT 1000
""",
    "Node conditions (0 = false, 1 = true)",
)

first = esql_rows(
    f"""
FROM {KUBELET_STREAM}
| WHERE k8s.cluster.name == "{CLUSTER}" AND k8s.node.memory.available IS NOT NULL
| STATS first_sample = MIN(@timestamp), last_sample = MAX(@timestamp)
| LIMIT 1
"""
)[0]
HISTORY_START = (
    parse_ts(BASELINE_FROM) if BASELINE_FROM else parse_ts(first["first_sample"])
)
history_hours = (parse_ts(first["last_sample"]) - HISTORY_START).total_seconds() / 3600
print(f"\nbaseline starts {HISTORY_START.isoformat()} ({history_hours:.1f} h so far)")

## 3. Create the per-node anomaly detection job

low_mean on k8s.node.memory.available, one model per node, 1 minute buckets. The datafeed starts at the first node sample. An existing job is reused.

In [ ]:
job_body = {
    "analysis_config": {
        "bucket_span": "1m",
        "detectors": [
            {
                "function": "low_mean",
                "field_name": "k8s.node.memory.available",
                "partition_field_name": "k8s.node.name",
            }
        ],
        "influencers": ["k8s.node.name"],
    },
    "analysis_limits": {"model_memory_limit": "32mb"},
    "data_description": {"time_field": "@timestamp"},
    "model_plot_config": {"enabled": True},
    "datafeed_config": {
        "datafeed_id": DATAFEED_ID,
        "indices": [KUBELET_STREAM],
        "query_delay": "20s",
        "frequency": "10s",
        "query": {
            "bool": {
                "filter": [
                    {"term": {"k8s.cluster.name": CLUSTER}},
                    {"exists": {"field": "k8s.node.memory.available"}},
                ]
            }
        },
    },
}

existing = es.options(ignore_status=404).ml.get_jobs(job_id=JOB_ID)
if existing.get("count", 0):
    print(f"job {JOB_ID} already exists, reusing it")
else:
    es.ml.put_job(job_id=JOB_ID, **job_body)
    print(f"created job {JOB_ID} with datafeed {DATAFEED_ID}")

stats = es.ml.get_job_stats(job_id=JOB_ID)["jobs"][0]
if stats["state"] != "opened":
    es.ml.open_job(job_id=JOB_ID)
feed = es.ml.get_datafeed_stats(datafeed_id=DATAFEED_ID)["datafeeds"][0]
if feed["state"] != "started":
    es.ml.start_datafeed(datafeed_id=DATAFEED_ID, start=HISTORY_START.isoformat())
    print(f"datafeed started from {HISTORY_START.isoformat()}")

# Make the job visible in Kibana > Machine Learning.
try:
    kbn("GET", "/api/ml/saved_objects/sync", params={"simulate": "false"})
except requests.HTTPError as exc:
    print(
        f"job sync skipped (HTTP {exc.response.status_code}): use Machine Learning > Synchronize saved objects"
    )

print("\nwaiting 60 s for the first buckets...")
time.sleep(60)
stats = es.ml.get_job_stats(job_id=JOB_ID)["jobs"][0]
print(
    f"state={stats['state']} processed_record_count={stats['data_counts']['processed_record_count']}"
    f" latest_record={stats['data_counts'].get('latest_record_timestamp')}"
)
print("Kibana:", f"{KBN_BASE}/app/ml/jobs?_a=(queryText:'id:{JOB_ID}')")

## 4. Create the anomaly detection rule

Record result type, severity 25, interim results off, checked every minute. Existing rules are reused.

In [ ]:
def find_rule(name):
    """Return the rule with this exact name and the lab tag, or None."""
    found = kbn(
        "GET",
        "/api/alerting/rules/_find",
        params={
            "search": f'"{name}"',
            "search_fields": "name",
            "per_page": 100,
            "filter": f'alert.attributes.tags:"{LAB_TAG}"',
        },
    )
    for rule in found.get("data", []):
        if rule["name"] == name:
            return rule
    return None


def ensure_rule(body):
    rule = find_rule(body["name"])
    if rule:
        print(f"rule exists: {rule['name']} ({rule['id']})")
        return rule
    rule = kbn("POST", "/api/alerting/rule", data=json.dumps(body))
    print(f"created rule: {rule['name']} ({rule['id']})")
    return rule


anomaly_rule = ensure_rule(
    {
        "name": RULE_ANOMALY,
        "rule_type_id": "xpack.ml.anomaly_detection_alert",
        "consumer": "alerts",
        "schedule": {"interval": "1m"},
        "tags": [LAB_TAG, "Kubernetes"],
        "params": {
            "jobSelection": {"jobIds": [JOB_ID], "groupIds": []},
            "severity": SEVERITY,
            "resultType": "record",
            "includeInterim": False,
            "lookbackInterval": None,
            "topNBuckets": None,
        },
        "actions": [],
    }
)
print(
    "Kibana:",
    f"{KBN_BASE}/app/management/insightsAndAlerting/triggersActions/rule/{anomaly_rule['id']}",
)

## 5. Create the headroom rule and the pressure condition rules

An Elasticsearch query rule on available memory below the headroom level, one alert per row, 2 minute window, checked every minute. The two condition rules use the `[Kubernetes OTel]` template queries filtered to this cluster, with the alert delay set to one match.

In [ ]:
HEADROOM_QUERY = f"""
FROM {KUBELET_STREAM}
| WHERE k8s.cluster.name == "{CLUSTER}"
    AND k8s.node.memory.available IS NOT NULL
| STATS available_gib = MIN(k8s.node.memory.available) / 1073741824.0
    BY k8s.cluster.name, k8s.node.name
| WHERE available_gib < {HEADROOM_GIB}
"""

headroom_rule = ensure_rule(
    {
        "name": RULE_HEADROOM,
        "rule_type_id": ".es-query",
        "consumer": "alerts",
        "schedule": {"interval": "1m"},
        "tags": [LAB_TAG, "Kubernetes"],
        "params": {
            "searchType": "esqlQuery",
            "esqlQuery": {"esql": HEADROOM_QUERY.strip()},
            "timeField": "@timestamp",
            "timeWindowSize": 2,
            "timeWindowUnit": "m",
            "size": 0,
            "threshold": [0],
            "thresholdComparator": ">",
            "groupBy": "row",
            "termField": "k8s.node.name",
            "termSize": 20,
            "excludeHitsFromPreviousRun": False,
        },
        "actions": [],
    }
)


def condition_query(field, status):
    return f"""
TS {CLUSTER_STREAM}
| WHERE {field} IS NOT NULL
| WHERE k8s.cluster.name == "{CLUSTER}"
| STATS any_pressure = MAX({field})
    BY k8s.node.name
| WHERE any_pressure == 1
| EVAL status = "{status}"
""".strip()


condition_rules = []
for name, field, status in (
    (RULE_MEMORY_PRESSURE, "k8s.node.condition_memory_pressure", "MemoryPressure"),
    (RULE_DISK_PRESSURE, "k8s.node.condition_disk_pressure", "DiskPressure"),
):
    condition_rules.append(
        ensure_rule(
            {
                "name": name,
                "rule_type_id": ".es-query",
                "consumer": "alerts",
                "schedule": {"interval": "1m"},
                "tags": [LAB_TAG, "Kubernetes"],
                "alert_delay": {"active": 1},
                "params": {
                    "searchType": "esqlQuery",
                    "esqlQuery": {"esql": condition_query(field, status)},
                    "timeField": "@timestamp",
                    "timeWindowSize": 1,
                    "timeWindowUnit": "m",
                    "size": 0,
                    "threshold": [0],
                    "thresholdComparator": ">",
                    "groupBy": "row",
                    "termField": "k8s.node.name",
                    "termSize": 20,
                    "excludeHitsFromPreviousRun": False,
                },
                "actions": [],
            }
        )
    )

print(
    "\nrules in Kibana:",
    f"{KBN_BASE}/app/management/insightsAndAlerting/triggersActions/rules",
)

## 6. Read the anomaly

Run from here after the scenario. Anomaly records for the worker, with the score the rule evaluated (`initial_record_score`) next to the current one, and the alerts the rules opened.

In [ ]:
def window_bounds():
    """Scenario window as datetimes."""
    if RANGE_FROM and RANGE_TO:
        return parse_ts(RANGE_FROM), parse_ts(RANGE_TO)
    now = datetime.now(timezone.utc)
    return now - timedelta(minutes=WINDOW_MINUTES), now


W_FROM, W_TO = window_bounds()
print(f"scenario window: {W_FROM.isoformat()} to {W_TO.isoformat()}\n")

records = es.ml.get_records(
    job_id=JOB_ID,
    start=W_FROM.isoformat(),
    end=W_TO.isoformat(),
    record_score=0,
    sort="timestamp",
    desc=False,
    page={"from": 0, "size": 500},
)["records"]
records = [r for r in records if r.get("partition_field_value") == NODE]
print(
    f"{'bucket (UTC)':20} {'score':>6} {'initial':>8} {'actual GiB':>11} {'typical GiB':>12}"
)
for r in records:
    if r["initial_record_score"] >= 3 or r["record_score"] >= 3:
        print(
            f"{fmt(parse_ts(r['timestamp'])):20} {r['record_score']:6.1f} {r['initial_record_score']:8.1f} "
            f"{r['actual'][0] / 1073741824:11.2f} {r['typical'][0] / 1073741824:12.2f}"
        )
print(
    f"({len(records)} records for {NODE} in the window; rows with a score under 3 hidden)"
)

alerts = es.options(ignore_status=404).search(
    index=".alerts-ml.anomaly-detection.alerts-*,.alerts-stack.alerts-*",
    query={
        "bool": {
            "filter": [
                {"term": {"kibana.alert.rule.tags": LAB_TAG}},
                {
                    "range": {
                        "kibana.alert.start": {
                            "gte": W_FROM.isoformat(),
                            "lte": W_TO.isoformat(),
                        }
                    }
                },
            ]
        }
    },
    sort=[{"kibana.alert.start": "asc"}],
    size=50,
    source=False,
    fields=[
        "kibana.alert.rule.name",
        "kibana.alert.start",
        "kibana.alert.status",
        "kibana.alert.instance.id",
    ],
)
ALERTS = {}
print(f"\n{'alert start (UTC)':20} {'status':10} rule")
for hit in alerts.get("hits", {}).get("hits", []):
    f = {k: v[0] for k, v in hit.get("fields", {}).items()}
    start = parse_ts(f["kibana.alert.start"])
    print(
        f"{fmt(start):20} {f.get('kibana.alert.status', ''):10} {f['kibana.alert.rule.name']}  [{f.get('kibana.alert.instance.id', '')}]"
    )
    ALERTS.setdefault(f["kibana.alert.rule.name"], start)
ANOMALY_ALERT = ALERTS.get(RULE_ANOMALY)
HEADROOM_ALERT = ALERTS.get(RULE_HEADROOM)
print(
    "\nAnomaly Explorer:",
    f"{KBN_BASE}/app/ml/explorer?_g=(ml:(jobIds:!('{JOB_ID}')),time:(from:'{W_FROM.isoformat()}',to:'{W_TO.isoformat()}'))",
)

## 7. Find the pod that changed and check the evictions

Each pod's last five minutes against its baseline, anchored two minutes after the anomaly alert (or at the end of the window if none opened). Then the Evicted events.

In [ ]:
anchor = (ANOMALY_ALERT + timedelta(minutes=2)) if ANOMALY_ALERT else W_TO
anchor_iso = anchor.isoformat().replace("+00:00", "Z")
print(f"pod change query anchored at {anchor_iso}")

esql(
    f"""
FROM {KUBELET_STREAM}
| WHERE k8s.cluster.name == "{CLUSTER}"
    AND k8s.node.name == "{NODE}"
    AND k8s.pod.memory.working_set IS NOT NULL
    AND @timestamp >= TO_DATETIME("{anchor_iso}") - 1 hour
    AND @timestamp <= TO_DATETIME("{anchor_iso}")
| STATS
    recent = AVG(k8s.pod.memory.working_set) WHERE @timestamp >= TO_DATETIME("{anchor_iso}") - 5 minutes,
    baseline = AVG(k8s.pod.memory.working_set) WHERE @timestamp < TO_DATETIME("{anchor_iso}") - 15 minutes
    BY k8s.namespace.name, k8s.pod.name
| EVAL recent_mib = ROUND(recent / 1048576, 1),
       baseline_mib = ROUND(COALESCE(baseline, 0) / 1048576, 1)
| EVAL change_mib = ROUND(recent_mib - baseline_mib, 1)
| WHERE recent_mib IS NOT NULL
| SORT change_mib DESC
| LIMIT 5
""",
    "Pods on the worker by memory change",
)

print("\nEviction order on the node (run in a terminal):")
print(f"  kubectl get pods -A --field-selector spec.nodeName={NODE} \\")
print(
    "    -o custom-columns=NS:.metadata.namespace,NAME:.metadata.name,QOS:.status.qosClass,PRIORITY:.spec.priority"
)

esql(
    f"""
FROM {EVENTS_STREAM}
| WHERE k8s.cluster.name == "{CLUSTER}"
    AND k8s.event.reason == "Evicted"
    AND {window()}
| KEEP @timestamp, k8s.object.name, k8s.event.reason, body.text
| SORT @timestamp
| LIMIT 1000
""",
    "Evicted events in the window",
)

## 8. Measure the warning time

Timeline of the scenario: load starts, first anomaly score, alerts, MemoryPressure on and off, first eviction, first failed request. Then how far ahead of MemoryPressure and of the first failed request the anomaly alert opened.

In [ ]:
def first_event(reason, names):
    rows = esql_rows(
        f"""
FROM {EVENTS_STREAM}
| WHERE k8s.cluster.name == "{CLUSTER}" AND k8s.event.reason == "{reason}"
    AND k8s.object.name IN ({", ".join(f'"{n}"' for n in names)})
    AND {window()}
| STATS first_seen = MIN(@timestamp), last_seen = MAX(@timestamp) BY k8s.object.name
| SORT first_seen
"""
    )
    return {
        r["k8s.object.name"]: (parse_ts(r["first_seen"]), parse_ts(r["last_seen"]))
        for r in rows
    }


started = first_event("Started", ["memory-change", "memory-escalation"])
killed = first_event("Killing", ["memory-change", "memory-escalation"])
evicted = first_event("Evicted", ["memory-change", "memory-escalation", "pressure-web"])
LOAD_START = min((t[0] for t in started.values()), default=None)
LOAD_REMOVED = max((t[1] for t in killed.values()), default=None)
FIRST_EVICTION = min((t[0] for t in evicted.values()), default=None)

# MemoryPressure transitions from the 10 s condition samples.
samples = esql_rows(
    f"""
FROM {CLUSTER_STREAM}
| WHERE k8s.cluster.name == "{CLUSTER}" AND k8s.node.name == "{NODE}"
    AND k8s.node.condition_memory_pressure IS NOT NULL AND {window()}
| KEEP @timestamp, k8s.node.condition_memory_pressure
| SORT @timestamp
| LIMIT 10000
"""
)
PRESSURE_TRUE = PRESSURE_CLEAR = None
for s in samples:
    ts, value = parse_ts(s["@timestamp"]), s["k8s.node.condition_memory_pressure"]
    if (
        PRESSURE_TRUE is None
        and value == 1
        and (LOAD_START is None or ts >= LOAD_START)
    ):
        PRESSURE_TRUE = ts
    elif (
        PRESSURE_TRUE is not None
        and PRESSURE_CLEAR is None
        and value == 0
        and ts > PRESSURE_TRUE
    ):
        PRESSURE_CLEAR = ts

# First finalized anomaly score after the load started, and the first one at or above the severity.
after_load = [
    r for r in records if LOAD_START is None or parse_ts(r["timestamp"]) >= LOAD_START
]
FIRST_SCORE = after_load[0]["initial_record_score"] if after_load else None
first_hit = next((r for r in after_load if r["initial_record_score"] >= SEVERITY), None)

# First probe export with a status other than 200 after the load started.
after_load_filter = ""
if LOAD_START:
    after_load_filter = f'AND @timestamp >= TO_DATETIME("{LOAD_START.isoformat().replace("+00:00", "Z")}")'
failed_rows = esql_rows(
    f"""
FROM {PROBE_STREAM}
| WHERE service.name == "node-pressure-probe" AND `attributes.test-id` == "{TEST_ID}"
    AND attributes.status != "200" AND {window()}
    {after_load_filter}
| STATS first_failed = MIN(@timestamp), failed_exports = COUNT(*)
"""
)
FIRST_FAILED = (
    parse_ts(failed_rows[0]["first_failed"])
    if failed_rows and failed_rows[0]["first_failed"]
    else None
)

rows = [
    (
        "Load starts",
        ", ".join(
            f"{fmt(t[0])} ({n})"
            for n, t in sorted(started.items(), key=lambda x: x[1][0])
        )
        or "none",
    ),
    (
        "First finalized anomaly score",
        f"{FIRST_SCORE:.1f}" if FIRST_SCORE is not None else "none",
    ),
    (
        "First record at or above severity",
        (
            f"{fmt(parse_ts(first_hit['timestamp']))} bucket, score {first_hit['initial_record_score']:.1f}"
            if first_hit
            else "none"
        ),
    ),
    ("Anomaly alert", fmt(ANOMALY_ALERT)),
    ("MemoryPressure true", fmt(PRESSURE_TRUE)),
    ("Headroom alert", fmt(HEADROOM_ALERT)),
    ("Memory pressure rule alert", fmt(ALERTS.get(RULE_MEMORY_PRESSURE))),
    ("First eviction", fmt(FIRST_EVICTION)),
    ("First failed request (export bucket)", fmt(FIRST_FAILED)),
    ("Load removed", fmt(LOAD_REMOVED)),
    ("MemoryPressure clears", fmt(PRESSURE_CLEAR)),
]
print(f"{'Event':38} UTC")
for label, value in rows:
    print(f"{label:38} {value}")


def lead(a, b):
    if a and b:
        seconds = int((b - a).total_seconds())
        sign = "-" if seconds < 0 else ""
        seconds = abs(seconds)
        return f"{sign}{seconds // 60} min {seconds % 60} s"
    return "n/a"


print()
print("anomaly alert before MemoryPressure:      ", lead(ANOMALY_ALERT, PRESSURE_TRUE))
print("anomaly alert before first failed request:", lead(ANOMALY_ALERT, FIRST_FAILED))
print("headroom alert after MemoryPressure:      ", lead(PRESSURE_TRUE, HEADROOM_ALERT))
print("grace period (MemoryPressure to eviction):", lead(PRESSURE_TRUE, FIRST_EVICTION))
print("MemoryPressure held after load removed:   ", lead(LOAD_REMOVED, PRESSURE_CLEAR))

In [ ]:
# Probe requests per minute by HTTP status.
try:
    esql(
        f"""
FROM {PROBE_STREAM}
| WHERE service.name == "node-pressure-probe" AND `attributes.test-id` == "{TEST_ID}"
    AND k6_http_reqs IS NOT NULL AND {window()}
| STATS requests = SUM(k6_http_reqs) BY minute = BUCKET(@timestamp, 1 minute), status = attributes.status
| SORT minute, status
| LIMIT 500
""",
        "Probe requests per minute by HTTP status (0 = connection failed)",
    )
except (
    Exception
) as exc:  # counter stored as a counter type: count export buckets instead
    esql(
        f"""
FROM {PROBE_STREAM}
| WHERE service.name == "node-pressure-probe" AND `attributes.test-id` == "{TEST_ID}"
    AND k6_http_req_duration IS NOT NULL AND {window()}
| STATS exports = COUNT(*) BY minute = BUCKET(@timestamp, 1 minute), status = attributes.status
| SORT minute, status
| LIMIT 500
""",
        "Probe export buckets per minute by HTTP status",
    )

esql(
    f"""
FROM {KUBELET_STREAM}
| WHERE k8s.cluster.name == "{CLUSTER}" AND k8s.node.name == "{NODE}"
    AND k8s.node.memory.available IS NOT NULL AND {window()}
| STATS available_gib = ROUND(MIN(k8s.node.memory.available) / 1073741824.0, 2)
    BY minute = BUCKET(@timestamp, 1 minute)
| SORT minute
| LIMIT 500
""",
    f"Worker available memory per minute (eviction at {EVICTION_GIB} GiB, headroom alert under {HEADROOM_GIB} GiB)",
)

## 9. Import the timeline dashboard

Imports `dashboard.ndjson`: KPIs, available memory, MemoryPressure, HTTP responses, working set by pod, and the job's model bounds. Skipped when the file is missing.

In [ ]:
import pathlib

ndjson = pathlib.Path("dashboard.ndjson")
DASHBOARD_ID = None
if ndjson.exists():
    response = requests.post(
        f"{KBN_BASE}/api/saved_objects/_import",
        params={"overwrite": "true"},
        headers={"Authorization": f"ApiKey {API_KEY}", "kbn-xsrf": "true"},
        files={"file": ("dashboard.ndjson", ndjson.read_bytes(), "application/ndjson")},
        timeout=120,
    )
    response.raise_for_status()
    result = response.json()
    imported = [o for o in result.get("successResults", []) if o["type"] == "dashboard"]
    if imported:
        DASHBOARD_ID = imported[0]["id"]
        print(
            f"imported dashboard {DASHBOARD_ID}: {KBN_BASE}/app/dashboards#/view/{DASHBOARD_ID}"
            f"?_g=(time:(from:'{W_FROM.isoformat()}',to:'{W_TO.isoformat()}'))"
        )
    else:
        print("import returned no dashboard:", json.dumps(result)[:500])
else:
    print("dashboard.ndjson not found next to the notebook, skipping")

## 10. Clean up

Deletes the four rules, the job with its datafeed, and the dashboard. `k8s/lab.sh down` deletes the cluster.

In [ ]:
for name in (RULE_ANOMALY, RULE_HEADROOM, RULE_MEMORY_PRESSURE, RULE_DISK_PRESSURE):
    rule = find_rule(name)
    if rule:
        response = requests.delete(
            f"{KBN_BASE}/api/alerting/rule/{rule['id']}",
            headers=KBN_HEADERS,
            timeout=120,
        )
        print(f"delete rule {name}: HTTP {response.status_code}")
    else:
        print(f"rule {name}: not found")

if globals().get("DASHBOARD_ID"):
    response = requests.delete(
        f"{KBN_BASE}/api/saved_objects/dashboard/{DASHBOARD_ID}",
        headers=KBN_HEADERS,
        timeout=120,
    )
    print(f"delete dashboard {DASHBOARD_ID}: HTTP {response.status_code}")

es.options(ignore_status=404).ml.stop_datafeed(datafeed_id=DATAFEED_ID, force=True)
es.options(ignore_status=404).ml.delete_datafeed(datafeed_id=DATAFEED_ID, force=True)
es.options(ignore_status=404).ml.close_job(job_id=JOB_ID, force=True)
response = es.options(ignore_status=404).ml.delete_job(
    job_id=JOB_ID, delete_user_annotations=True, force=True
)
print(
    f"delete job {JOB_ID} and datafeed {DATAFEED_ID}: {response.get('acknowledged', response)}"
)
print("done")